# ALL Met Painting Eyes

## The Met API

https://metmuseum.github.io/

https://github.com/metmuseum/openaccess

**Please limit request rate to 80 requests per second.**

In [ ]:
from google.colab import userdata
PAT_XYZ = userdata.get("PAT_XYZ")

In [ ]:
!pip install mediapipe
!pip install ultralytics
!wget https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task
!mkdir json && wget -P json https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/json/mp_masks_definitions.json
!wget https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/utils.py
!wget https://raw.githubusercontent.com/acervos-digitais/met-faces-utils/refs/heads/main/utils_paintings.py
!git clone https://{PAT_XYZ}@github.com/acervos-digitais/met-faces-data.git data
!cd data && git config user.name "Thiago Hersan" && git config user.email "thiago.hersan+github@gmail.com"

In [ ]:
from utils import get_combined_jsons

from utils_paintings import PaintingsUtils

DATA_DIR = "./data"
JSON_DIR = f"{DATA_DIR}/json"
IMG_DIR = f"{DATA_DIR}/image"

## Painting Objects

- $15\text{,}178$ on website
- $15\text{,}050$ (±100) available in API
- $14\text{,}233$ (±50) have images (according to `hasImages=true` API query param)
- $9\text{,}015$ ($63\%$) actually have images that can be downloaded
- $\sim6\text{,}000$ ($66\%$) have faces
- $\sim5\text{,}400$ ($90\%$) have extractable eye locations/shapes

In [ ]:
obj_ids = PaintingsUtils.get_object_ids()
len(obj_ids)

## Get Object Metadata

In [ ]:
mPU = PaintingsUtils(JSON_DIR, IMG_DIR)

for cnt,oid in enumerate(obj_ids):
  if cnt % 16 == 0:
    print(f"{cnt} / {len(obj_ids)}")

  mPU.get_obj_data(oid)

## Get Faces, Landmarks and Eyes

In [ ]:
mPU = PaintingsUtils(JSON_DIR, IMG_DIR, with_detectors=True)
objects_data = get_combined_jsons(mPU.json_objs_dir)
len(objects_data)

In [ ]:
for cnt,obj_data in enumerate(objects_data):
  if cnt % 16 == 0:
    print(f"{cnt} / {len(objects_data)}")

  if mPU.is_done(obj_data):
    continue

  img = mPU.get_image(obj_data["primaryImage"])
  if img is None:
    continue

  face_data = mPU.get_face_data(obj_data, img)
  if face_data is None:
    continue

  landmark_data = mPU.get_landmark_data(face_data, img)
  if landmark_data is None:
    continue

  mPU.get_eye_images(landmark_data, img)

## Checks

In [ ]:
mPU = PaintingsUtils(JSON_DIR, IMG_DIR)
objs = get_combined_jsons(mPU.json_objs_dir)
faces = get_combined_jsons(mPU.json_faces_dir)
landmarks = get_combined_jsons(mPU.json_landmarks_dir)

no_imgs_s = set(mPU.no_imgs)
ye_imgs_s = set([o["objectID"] for o in objs])

no_faces_s = set(mPU.no_faces)
ye_faces_s = set([o["objectID"] for o in faces])

no_lands_s = set(mPU.no_landmarks)
ye_lands_s = set([o["objectID"] for o in landmarks])

print(len(no_imgs_s), len(ye_imgs_s), len(no_imgs_s) + len(ye_imgs_s))
print(len(no_faces_s), len(ye_faces_s), len(no_faces_s) + len(ye_faces_s))
print(len(no_lands_s), len(ye_lands_s), len(no_lands_s) + len(ye_lands_s))

print(len(no_imgs_s.intersection(ye_imgs_s)))
print(len(no_faces_s.intersection(ye_faces_s)))
print(len(no_lands_s.intersection(ye_lands_s)))

## Export csv

In [ ]:
mPU = PaintingsUtils(JSON_DIR, IMG_DIR)
objs = get_combined_jsons(mPU.json_objs_dir)
faces = get_combined_jsons(mPU.json_faces_dir)
landmarks = get_combined_jsons(mPU.json_landmarks_dir)

In [ ]:
import pandas as pd

KEEP_COLS = [
  "objectID",
  "accessionNumber",
  "artistDisplayName",
  "title",
  "objectDate",
  "medium",
  "dimensions",
  "measurements.Height",
  "measurements.Width",
  "department",
  "primaryImage",
]

CSV_DIR = f"{DATA_DIR}/csv"

objs_df = pd.json_normalize(objs)[KEEP_COLS]
faces_df = pd.json_normalize(faces)[KEEP_COLS]
landmarks_df = pd.json_normalize(landmarks)[KEEP_COLS]

In [ ]:
objs_df.to_csv(f"{CSV_DIR}/objects.csv", index=False)
faces_df.to_csv(f"{CSV_DIR}/faces.csv", index=False)
landmarks_df.to_csv(f"{CSV_DIR}/landmarks.csv", index=False)